In [1]:
import pandas as pd

df = pd.read_csv('../data/zillow.csv')

display(df.head(5))

,zpid,state_code,state_name,address,street,city,zipcode,price,price_formatted,beds,...,latitude,longitude,status,home_type,days_on_zillow,zestimate,detail_url,has_open_house,open_house_date,is_featured
0,17264897,CA,California,"979 Kevin Ave, Redlands, CA 92373",979 Kevin Ave,Redlands,92373,447000,"$447,000",3.0,...,34.040520,-117.186195,House for sale,SINGLE_FAMILY,5,455500.0,https://www.zillow.com/homedetails/979-Kevin-A...,False,NaN,False
1,20021372,CA,California,"13114 Addison St, Sherman Oaks, CA 91423",13114 Addison St,Sherman Oaks,91423,2795000,"$2,795,000",4.0,...,34.160885,-118.418770,House for sale,SINGLE_FAMILY,8,NaN,https://www.zillow.com/homedetails/13114-Addis...,True,2026-02-08T12:00:00,False
2,20009320,CA,California,"6032 Goodland Ave, North Hollywood, CA 91606",6032 Goodland Ave,North Hollywood,91606,1718000,"$1,718,000",4.0,...,34.180450,-118.411280,House for sale,SINGLE_FAMILY,8,1695500.0,https://www.zillow.com/homedetails/6032-Goodla...,False,NaN,False
3,460204882,CA,California,"3325 Tonopah St, Oceanside, CA 92054",3325 Tonopah St,Oceanside,92054,899999,"$899,999",3.0,...,33.214260,-117.341500,Coming soon,SINGLE_FAMILY,4,NaN,https://www.zillow.com/homedetails/3325-Tonopa...,False,NaN,False
4,20769150,CA,California,"963 Pine Grove Ave, Los Angeles, CA 90042",963 Pine Grove Ave,Los Angeles,90042,995000,"$995,000",2.0,...,34.132890,-118.186220,House for sale,SINGLE_FAMILY,5,999900.0,https://www.zillow.com/homedetails/963-Pine-Gr...,False,NaN,False


In [2]:
df.columns

Index(['zpid', 'state_code', 'state_name', 'address', 'street', 'city',
       'zipcode', 'price', 'price_formatted', 'beds', 'baths', 'area_sqft',
       'latitude', 'longitude', 'status', 'home_type', 'days_on_zillow',
       'zestimate', 'detail_url', 'has_open_house', 'open_house_date',
       'is_featured'],
      dtype='object')

Initial issue is that state and state code cols do not match the information on the address.  Zipcode col is missing leading 0s

To resolve:
* Delete state_name, and state_code, street, city, and zipcode
* Manually split them cleanly

In [3]:
# dropping address related cols since many have missing information
# dropping price formatted since '$' is not useful for analysis
# dropping open_house_date since most rows are NaN

drop_cols = ['state_code', 'state_name', 'street', 'city', 'zipcode', 'price_formatted', 'open_house_date']

df.drop(columns=drop_cols, inplace=True)

display(df.head())

,zpid,address,price,beds,baths,area_sqft,latitude,longitude,status,home_type,days_on_zillow,zestimate,detail_url,has_open_house,is_featured
0,17264897,"979 Kevin Ave, Redlands, CA 92373",447000,3.0,2.0,1300.0,34.040520,-117.186195,House for sale,SINGLE_FAMILY,5,455500.0,https://www.zillow.com/homedetails/979-Kevin-A...,False,False
1,20021372,"13114 Addison St, Sherman Oaks, CA 91423",2795000,4.0,5.0,2900.0,34.160885,-118.418770,House for sale,SINGLE_FAMILY,8,NaN,https://www.zillow.com/homedetails/13114-Addis...,True,False
2,20009320,"6032 Goodland Ave, North Hollywood, CA 91606",1718000,4.0,3.0,3025.0,34.180450,-118.411280,House for sale,SINGLE_FAMILY,8,1695500.0,https://www.zillow.com/homedetails/6032-Goodla...,False,False
3,460204882,"3325 Tonopah St, Oceanside, CA 92054",899999,3.0,3.0,1682.0,33.214260,-117.341500,Coming soon,SINGLE_FAMILY,4,NaN,https://www.zillow.com/homedetails/3325-Tonopa...,False,False
4,20769150,"963 Pine Grove Ave, Los Angeles, CA 90042",995000,2.0,2.0,1271.0,34.132890,-118.186220,House for sale,SINGLE_FAMILY,5,999900.0,https://www.zillow.com/homedetails/963-Pine-Gr...,False,False


**Correction of 'state_name' and 'state_code' columns**

Numerous columns possessed the incorrect state name and code, not matching the address column
Manually inspecting some of the incorrect data links showed the address column is correct while the original split columns possessed incorrect information


In [4]:
# Map state codes to full names
state_map = {
    'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas',
    'CA': 'California', 'CO': 'Colorado', 'CT': 'Connecticut', 'DE': 'Delaware',
    'FL': 'Florida', 'GA': 'Georgia', 'HI': 'Hawaii', 'ID': 'Idaho',
    'IL': 'Illinois', 'IN': 'Indiana', 'IA': 'Iowa', 'KS': 'Kansas',
    'KY': 'Kentucky', 'LA': 'Louisiana', 'ME': 'Maine', 'MD': 'Maryland',
    'MA': 'Massachusetts', 'MI': 'Michigan', 'MN': 'Minnesota', 'MS': 'Mississippi',
    'MO': 'Missouri', 'MT': 'Montana', 'NE': 'Nebraska', 'NV': 'Nevada',
    'NH': 'New Hampshire', 'NJ': 'New Jersey', 'NM': 'New Mexico', 'NY': 'New York',
    'NC': 'North Carolina', 'ND': 'North Dakota', 'OH': 'Ohio', 'OK': 'Oklahoma',
    'OR': 'Oregon', 'PA': 'Pennsylvania', 'RI': 'Rhode Island', 'SC': 'South Carolina',
    'SD': 'South Dakota', 'TN': 'Tennessee', 'TX': 'Texas', 'UT': 'Utah',
    'VT': 'Vermont', 'VA': 'Virginia', 'WA': 'Washington', 'WV': 'West Virginia',
    'WI': 'Wisconsin', 'WY': 'Wyoming', 'DC': 'District of Columbia'
}

In [5]:
address_parts = df['address'].str.split(', ', expand =True)

df['street_add'] = address_parts[0]
df['city'] = address_parts[1]
df['state_zipcode'] = address_parts[2]

state_zip_split = df['state_zipcode'].str.split(' ', expand =True)
df['state_code'] = state_zip_split[0]
df['zipcode'] = state_zip_split[1]

df['state_name'] = df['state_code'].map(state_map)

In [6]:
display(df.head())

,zpid,address,price,beds,baths,area_sqft,latitude,longitude,status,home_type,...,zestimate,detail_url,has_open_house,is_featured,street_add,city,state_zipcode,state_code,zipcode,state_name
0,17264897,"979 Kevin Ave, Redlands, CA 92373",447000,3.0,2.0,1300.0,34.040520,-117.186195,House for sale,SINGLE_FAMILY,...,455500.0,https://www.zillow.com/homedetails/979-Kevin-A...,False,False,979 Kevin Ave,Redlands,CA 92373,CA,92373,California
1,20021372,"13114 Addison St, Sherman Oaks, CA 91423",2795000,4.0,5.0,2900.0,34.160885,-118.418770,House for sale,SINGLE_FAMILY,...,NaN,https://www.zillow.com/homedetails/13114-Addis...,True,False,13114 Addison St,Sherman Oaks,CA 91423,CA,91423,California
2,20009320,"6032 Goodland Ave, North Hollywood, CA 91606",1718000,4.0,3.0,3025.0,34.180450,-118.411280,House for sale,SINGLE_FAMILY,...,1695500.0,https://www.zillow.com/homedetails/6032-Goodla...,False,False,6032 Goodland Ave,North Hollywood,CA 91606,CA,91606,California
3,460204882,"3325 Tonopah St, Oceanside, CA 92054",899999,3.0,3.0,1682.0,33.214260,-117.341500,Coming soon,SINGLE_FAMILY,...,NaN,https://www.zillow.com/homedetails/3325-Tonopa...,False,False,3325 Tonopah St,Oceanside,CA 92054,CA,92054,California
4,20769150,"963 Pine Grove Ave, Los Angeles, CA 90042",995000,2.0,2.0,1271.0,34.132890,-118.186220,House for sale,SINGLE_FAMILY,...,999900.0,https://www.zillow.com/homedetails/963-Pine-Gr...,False,False,963 Pine Grove Ave,Los Angeles,CA 90042,CA,90042,California


**At this point, the address parts have been corrected**
Example: There should not be any cols that say 500 Commonwealth Ave, Boston, MA 02135, but have state_name and state_code as Texas and TX

In [15]:
df.to_csv('../data/zillow_corrected_address.csv') # save the new df if desired

**Removal of home_type rows that are not single family homes**

In [9]:
df['home_type'].value_counts()

home_type
SINGLE_FAMILY    2108
CONDO              30
TOWNHOUSE          25
MANUFACTURED       23
MULTI_FAMILY       19
LOT                 9
Name: count, dtype: int64

In [10]:
df = (
    df[~df['home_type']
       .isin(['LOT', 'MANUFACTURED', 'MULTI_FAMILY']
        )
    ])

# confirms selected home_type rows are gone
df['home_type'].value_counts()

home_type
SINGLE_FAMILY    2108
CONDO              30
TOWNHOUSE          25
Name: count, dtype: int64

In [11]:
df.isnull().sum()

zpid                 0
address              0
price                0
beds                 0
baths                0
area_sqft           50
latitude             4
longitude            4
status               0
home_type            0
days_on_zillow       0
zestimate         1078
detail_url           0
has_open_house       0
is_featured          0
street_add           0
city                 0
state_zipcode        1
state_code           1
zipcode              1
state_name           1
dtype: int64

**Summary of Missing Values**
The one home missing state_zipcode, state_code, zipcode, state_name is zpid: 444753560
zpid: 444753560 is a new construction of homes, not one home, this needs to be removed 

area_sqft           50
latitude             4
longitude            4
zestimate         1078
state_zipcode        1
state_code           1
zipcode              1
state_name           1
dtype: int64

In [12]:
display(df[df['zipcode'].isna()])

,zpid,address,price,beds,baths,area_sqft,latitude,longitude,status,home_type,...,zestimate,detail_url,has_open_house,is_featured,street_add,city,state_zipcode,state_code,zipcode,state_name
2150,444753560,"Parker Plan, Bridle Creek",365900,3.0,2.0,1700.0,39.486683,-77.93531,New construction,SINGLE_FAMILY,...,NaN,https://www.zillow.com/community/bridle-creek/...,False,False,Parker Plan,Bridle Creek,None,None,None,NaN


In [13]:
# drop row since this is not an actual address
df.drop(
    df[
        df['zpid'] == 444753560].index, inplace=True
    )

**Missing Values After Removal of zpid=444753560**

area_sqft           50\
latitude             4\
longitude            4\
zestimate         1077

**Next Steps: Ensure lat/lon coords are correct.**

It is assumed that some values are incorrect because of previous address issues.

**Manual inspection found a few more unusable addresses**

* 455353390	(undisclosed Address), Seattle, WA 98112
* 460335453	Undisclosed, Canyon Creek, MT 59633
* 460335454	Undisclosed, Darby, MT 59829

Requires manual correction for lat/lon:
* 460321976	9033 Chapel Heights Rd, Auburn, AL 36830

In [14]:
# drops rows with undisclosed locations (high value properties)
drop_list = [455353390, 460335453, 460335454]

df = df[~df['zpid'].isin(drop_list)]

df.isnull().sum()

zpid                 0
address              0
price                0
beds                 0
baths                0
area_sqft           50
latitude             1
longitude            1
status               0
home_type            0
days_on_zillow       0
zestimate         1074
detail_url           0
has_open_house       0
is_featured          0
street_add           0
city                 0
state_zipcode        0
state_code           0
zipcode              0
state_name           0
dtype: int64

In [16]:
df.to_csv('../data/zillow_cleaned.csv') # save the new df if desired